# Elastic constants of MgO

*Based on Problem 7 of the [ICTP-MARVEL College 2026](https://github.com/marvel-nccr/ictp-marvel-college-2026) day-01 exercise, originally authored by Edward Linscott.*

Elastic constants describe how a material responds to small deformations. Hooke's law for a continuous medium is

$$\sigma_{ij} = \sum_{kl} C_{ijkl}\, \epsilon_{kl},$$

where $\sigma_{ij}$ is the stress tensor, $\epsilon_{kl}$ is the strain tensor, and $C_{ijkl}$ is the **stiffness tensor**. Symmetry reduces the 81 independent components of $C_{ijkl}$ to 21, and the **cubic symmetry** of MgO (rocksalt structure) reduces them further to just three:

$$
\mathbf{C} = \begin{pmatrix}
C_{11} & C_{12} & C_{12} & 0 & 0 & 0 \\
C_{12} & C_{11} & C_{12} & 0 & 0 & 0 \\
C_{12} & C_{12} & C_{11} & 0 & 0 & 0 \\
0 & 0 & 0 & C_{44} & 0 & 0 \\
0 & 0 & 0 & 0 & C_{44} & 0 \\
0 & 0 & 0 & 0 & 0 & C_{44}
\end{pmatrix}
$$

using **Voigt notation** (1=*xx*, 2=*yy*, 3=*zz*, 4=*yz*, 5=*zx*, 6=*xy*).

The elastic energy stored by a small strain $\{e_\mu\}$ is, to second order in the strain,

$$\Delta E = \frac{V_0}{2} \sum_{\mu\nu} C_{\mu\nu}\, e_\mu\, e_\nu,$$

where $V_0$ is the equilibrium cell volume. For MgO's cubic symmetry this reduces to

$$\frac{\Delta E}{V_0} = \frac{C_{11}}{2}(e_1^2 + e_2^2 + e_3^2) + C_{12}(e_1 e_2 + e_2 e_3 + e_3 e_1) + \frac{C_{44}}{2}(e_4^2 + e_5^2 + e_6^2).$$

The bulk modulus is related to the elastic constants by

$$B = \tfrac{1}{3}(C_{11} + 2C_{12}),$$

which links this notebook directly to the EOS results from [qe_eos_bulkmodulus.ipynb](qe_eos_bulkmodulus.ipynb).

We compute $C_{11}$ and $C_{12}$ from an **orthorhombic** volume-conserving strain (Problem 7B), and $C_{44}$ from a **monoclinic shear** strain (Problem 7C).

In [ ]:
# ── Bootstrap condacolab (triggers kernel restart on first run) ────────────────
# After the restart, re-run from this cell — it will skip the install.
try:
    import condacolab
    condacolab.check()
    print('✅ condacolab active — continue to next cell')
except Exception:
    import subprocess, sys
    print('Installing condacolab …')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'],
        stdout=subprocess.DEVNULL,
    )
    import condacolab
    condacolab.install()    # ← kernel restarts here; re-run this cell after restart

In [ ]:
# ── Install QE environment and clone tutorial repo ────────────────────────────
# Takes ~5–10 minutes on first run; skipped automatically on re-run.
import condacolab, subprocess, os, sys, glob
from pathlib import Path
condacolab.check()

ENV_NAME = 'qe_env'
REPO_URL = 'https://github.com/pietrodelugas/qe_with_notebooks.git'
REPO_DIR = '/content/qe_with_notebooks'

# Create qe_env if not already present
_env_bin = f'/usr/local/envs/{ENV_NAME}/bin'
if not os.path.isdir(_env_bin):
    print(f"Creating '{ENV_NAME}' with QE 7.5 — takes ~5–10 minutes …")
    subprocess.run(
        ['conda', 'create', '-n', ENV_NAME, '-c', 'conda-forge', '--yes',
         'python=3.12', 'qe=7.5', 'numpy', 'matplotlib', 'ase', 'scipy'],
        check=True,
    )
    subprocess.run(
        ['conda', 'run', '-n', ENV_NAME, 'pip', 'install', '-q', 'ovito'],
        check=True,
    )
    print(f"✅ '{ENV_NAME}' ready")

# Clone tutorial repo (modules + pseudos)
if not os.path.isdir(REPO_DIR):
    print('Cloning tutorial repo …')
    subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', 'distro', REPO_URL, REPO_DIR],
        check=True,
    )
    print('✅ Repo cloned')

# Expose qe_env Python packages to this interpreter
for sp in glob.glob(f'/usr/local/envs/{ENV_NAME}/lib/python*/site-packages'):
    if sp not in sys.path:
        sys.path.insert(0, sp)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify
print('\nQE executables:')
for exe in ['pw.x', 'bands.x', 'dos.x', 'projwfc.x']:
    p = Path(_env_bin) / exe
    print(f"  {'✅' if p.is_file() else '❌'}  {exe}")
print('\nPython packages:')
for pkg in ['numpy', 'matplotlib', 'ase', 'scipy', 'pw_input']:
    try:
        __import__(pkg)
        print(f'  ✅  {pkg}')
    except ImportError as e:
        print(f'  ❌  {pkg} — {e}')
print('\n🎉 Ready!')

In [ ]:
from pathlib import Path
import numpy as np
from ase.build import bulk

from convergence_runner import QERunner, RY_TO_EV
from elastic_tools import (
    build_ortho_input, build_mono_input,
    fit_elastic_quad, extract_c11_c12, extract_c44,
)
import os
os.environ['OMP_NUM_THREADS'] = '1'

QE_BIN = Path('/usr/local/envs/qe_env/bin')
RUN_ROOT   = Path('/content')
PSEUDO_DIR = Path(REPO_DIR) / 'pseudo'
ELAST_DIR  = RUN_ROOT / 'out' / 'elastic'
ELAST_DIR.mkdir(parents=True, exist_ok=True)

PW_CMD = [str(QE_BIN / 'pw.x')]

PSEUDOS = {
    'Mg': 'Mg.upf',
    'O':  'O.upf',
}

# ---- Results from qe_eos_bulkmodulus.ipynb (Birch-Murnaghan fit) ----------
# Fill in your values here before running the elastic constant sweeps.
A0_ANG       = 4.21    # Å  — equilibrium lattice parameter
B0_GPA       = 160.0   # GPa — bulk modulus
V0_PRIM_ANG3 = (A0_ANG**3) / 4   # Å³ — primitive cell volume (2 atoms)
V0_CONV_ANG3 = 4 * V0_PRIM_ANG3  # Å³ — conventional cell volume (8 atoms)

ECUTWFC_CONV = 60   # Ry  — from convergence notebook
NK_ELASTIC   = 4    # k-grid along each direction (equivalent density to nk=8 for primitive cell)
FORCE_RERUN  = False

runner = QERunner(PW_CMD)

print(f'a₀ = {A0_ANG:.4f} Å,  B₀ = {B0_GPA:.1f} GPa')
print(f'V₀ (primitive) = {V0_PRIM_ANG3:.4f} Å³,  V₀ (conventional, 8 atoms) = {V0_CONV_ANG3:.4f} Å³')

## Problem 7A: Primitive cell vs conventional cell

The elastic constant calculations use the **8-atom conventional cubic cell** rather than the 2-atom FCC primitive cell. What are the trade-offs?

<details>
<summary><b>Solution</b></summary>

**Advantages of the primitive cell:**

- Computationally cheaper: the cost of DFT scales as $\mathcal{O}(N^3)$ in the number of electrons, so a 4× larger cell is roughly 64× more expensive per **k**-point.
- The primitive cell exposes more translational symmetry, which `pw.x` uses to reduce the eigenproblem size.

**Advantages of the conventional cell for elastic constants:**

- The cubic point-group symmetries ($C_4$ rotations about $x$, $y$, $z$) act manifestly on orthogonal Cartesian axes, making it straightforward to apply Cartesian strain patterns.
- The smaller Brillouin zone (4× smaller) means a coarser **k**-grid achieves the same sampling density — partly offsetting the larger cell cost. In fact this notebook uses `NK_ELASTIC = 4` (a 4×4×4 grid), equivalent in density to an 8×8×8 grid for the primitive cell.
- For the orthorhombic and monoclinic strain patterns used here, `ibrav=8` and `ibrav=12` require orthogonal or nearly-orthogonal cell vectors, which arise naturally from the conventional cell.

> [!NOTE]
> Because we apply strain to the **equilibrium** geometry, it is essential to use the lattice parameter $a_0$ from the Birch–Murnaghan fit in the previous notebook, not the experimental value.

</details>

## Problem 7B: $C_{11}$ and $C_{12}$ — orthorhombic strain

### Part A: Deriving the energy–strain relation

Apply the **volume-conserving orthorhombic strain**

$$\varepsilon = \begin{pmatrix} x & 0 & 0 \\ 0 & -x & 0 \\ 0 & 0 & \dfrac{x^2}{1-x^2} \end{pmatrix}$$

to the cubic equilibrium cell. The deformed lattice vectors $\mathbf{a}'_i = \mathbf{a}_j\,(I+\varepsilon)_{ji}$ have lengths

$$|\mathbf{a}'_1| = a_0(1+x), \quad |\mathbf{a}'_2| = a_0(1-x), \quad |\mathbf{a}'_3| = \frac{a_0}{1-x^2},$$

giving `ibrav=8` celldm values

$$\text{celldm}(1) = \frac{a_0(1+x)}{a_\text{B}}, \qquad
  \text{celldm}(2) = \frac{1-x}{1+x}, \qquad
  \text{celldm}(3) = \frac{1}{(1-x^2)(1+x)},$$

where $a_\text{B} = 0.529177$ Å is the Bohr radius. To second order in $x$, the energy change is

$$\Delta E(x) \equiv E(x) - E(0) = V_0\,(C_{11} - C_{12})\,x^2.$$

Fitting $\Delta E(x) = \alpha x^2$ and combining with $B = \tfrac{1}{3}(C_{11}+2C_{12})$ from the EOS notebook:

$$C_{11} = \frac{2\alpha/V_0 + 3B}{3}, \qquad C_{12} = \frac{3B - \alpha/V_0}{3}.$$

> [!NOTE]
> We use `calculation = 'relax'` rather than `scf` because the strained orthorhombic cell has lower symmetry than the cubic one. Internal degrees of freedom that were fixed by symmetry in the cubic cell may now be free to relax, and the energy must be minimised with respect to all atomic positions before the elastic constant is meaningful.

> [!TIP]
> Choose strain amplitudes small enough that the quadratic approximation holds ($|x| \lesssim 0.05$) but large enough that $\Delta E$ is well above numerical noise. We sample $x \in \{-0.04, -0.02, +0.02, +0.04\}$ — symmetric about zero so that odd-order anharmonic terms cancel in the fit.

> [!NOTE]
> The sweep includes $x = 0$ to provide a clean reference energy $E(0)$ from the same `relax` run. At fitting time, however, we **exclude** it: $\Delta E(0) = 0$ by construction — it is the reference energy itself, not a measurement of the curvature. Including it would add the trivially-satisfied constraint $(0,\,0)$ to the fit without contributing any information. (With the forced-zero-intercept estimator used here, adding it does not change the numerical result either way, but the exclusion makes the logic transparent.)

<details>
<summary>Implementation: <code>build_ortho_input</code> (<code>elastic_tools.py</code>)</summary>

```python
def build_ortho_input(a0_ang, x, ecutwfc, nk, prefix, pseudo_dir, outdir, pseudos):
    atoms = bulk('MgO', 'rocksalt', a=a0_ang, cubic=True)
    control   = ControlNamelist(calculation='relax', ..., tprnfor=True, tstress=True)
    system    = SystemNamelist(ibrav=8, nat=8, ntyp=2, ecutwfc=ecutwfc,
                               celldm_1 = a0_ang * (1 + x) * _ANG_TO_BOHR,
                               celldm_2 = (1 - x) / (1 + x),
                               celldm_3 = 1 / ((1 - x**2) * (1 + x)))
    electrons = ElectronsNamelist(conv_thr=1.e-9)
    ions      = IonsNamelist()
    ...
    return PWInput(control, system, electrons, species, positions, kpoints, ions=ions)
```

</details>

In [ ]:
import matplotlib.pyplot as plt

STRAIN_VALUES = np.array([...])  # fill in symmetric strain amplitudes, e.g. ±0.02, ±0.04

ortho_cases = [
    (f'ortho_{i:02d}', build_ortho_input(
        A0_ANG, x, ECUTWFC_CONV, NK_ELASTIC,
        prefix=f'mgo_ortho_{i:02d}',
        pseudo_dir=PSEUDO_DIR, outdir=ELAST_DIR, pseudos=PSEUDOS,
    ))
    for i, x in enumerate(STRAIN_VALUES)
]

ortho_results = runner.run_sweep(ortho_cases, ELAST_DIR, force_rerun=FORCE_RERUN)

E_ortho  = np.array([r['energy_ry'] * RY_TO_EV for r in ortho_results])
E0_ortho = E_ortho[2]           # x = 0 is at index 2
dE_ortho = E_ortho - E0_ortho

print(f"{'x':>8}  {'E (eV/cell)':>16}  {'ΔE (meV/cell)':>14}")
for x, e, de in zip(STRAIN_VALUES, E_ortho, dE_ortho):
    print(f'{x:8.4f}  {e:16.6f}  {de*1000:14.4f}')

In [ ]:
mask         = STRAIN_VALUES != 0.0
coeff_ortho  = fit_elastic_quad(STRAIN_VALUES[mask], dE_ortho[mask])
C11, C12     = extract_c11_c12(coeff_ortho, V0_CONV_ANG3, B0_GPA)

C11_EXP, C12_EXP = 297.0, 95.0   # GPa — experiment (Karki et al. 1997)

print(f"{'':20}  {'C₁₁ (GPa)':>10}  {'C₁₂ (GPa)':>10}")
print(f"{'DFT (this work)':20}  {C11:10.1f}  {C12:10.1f}")
print(f"{'Experiment':20}  {C11_EXP:10.1f}  {C12_EXP:10.1f}")
print(f"\nCheck:  B = (C₁₁+2C₁₂)/3 = {(C11+2*C12)/3:.1f} GPa  (EOS: {B0_GPA:.1f} GPa)")

x_fine = np.linspace(STRAIN_VALUES.min(), STRAIN_VALUES.max(), 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(STRAIN_VALUES[mask], dE_ortho[mask] * 1000, zorder=3, label='DFT (relax)')
ax.plot(x_fine, coeff_ortho * x_fine**2 * 1000,
        label=f'Fit  $\\alpha = {coeff_ortho*1000:.1f}$ meV')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Strain amplitude $x$')
ax.set_ylabel('$\\Delta E$ (meV/cell)')
ax.set_title('MgO — orthorhombic strain')
ax.legend()
fig.tight_layout()
plt.show()

## Problem 7C: $C_{44}$ — monoclinic shear strain

### Part A: Deriving the energy–strain relation

Apply the **volume-conserving monoclinic shear strain**

$$\varepsilon = \begin{pmatrix} 0 & x/2 & 0 \\ x/2 & 0 & 0 \\ 0 & 0 & \dfrac{x^2}{4-x^2} \end{pmatrix}$$

The shear mixes the $x$ and $y$ directions while the $z$ axis elongates to preserve volume. The strained cell is monoclinic (`ibrav=12`, unique axis $c$), with

$$
\text{celldm}(1) = \frac{a_0\sqrt{1+x^2/4}}{a_\text{B}}, \quad
\text{celldm}(2) = 1, \quad
\text{celldm}(3) = \frac{4}{(4-x^2)\sqrt{1+x^2/4}}, \quad
\cos\gamma = \text{celldm}(4) = \frac{x}{1+x^2/4}.
$$

To second order in $x$ the energy change is

$$\Delta E(x) = \tfrac{1}{2}\,V_0\,C_{44}\,x^2,$$

so fitting $\Delta E(x) = \alpha x^2$ gives

$$C_{44} = \frac{2\alpha}{V_0}.$$

> [!NOTE]
> The monoclinic cell also requires `calculation = 'relax'`: the shear breaks the equivalence between the two Mg sublattice sites (and the two O sites) that existed in the cubic cell, so atomic positions must be allowed to relax.

> [!TIP]
> Use the same strain amplitudes as Problem 7B: $x \in \{-0.04, -0.02, 0.00, +0.02, +0.04\}$. The $x=0$ point again provides the reference energy and is excluded from the fit.

<details>
<summary>Implementation: <code>build_mono_input</code> (<code>elastic_tools.py</code>)</summary>

```python
def build_mono_input(a0_ang, x, ecutwfc, nk, prefix, pseudo_dir, outdir, pseudos):
    atoms = bulk('MgO', 'rocksalt', a=a0_ang, cubic=True)
    f = 1 + x**2 / 4   # shorthand; appears in all four celldm formulas
    control   = ControlNamelist(calculation='relax', ..., tprnfor=True, tstress=True)
    system    = SystemNamelist(ibrav=12, nat=8, ntyp=2, ecutwfc=ecutwfc,
                               celldm_1 = a0_ang * np.sqrt(f) * _ANG_TO_BOHR,
                               celldm_2 = 1.0,
                               celldm_3 = 4 / ((4 - x**2) * np.sqrt(f)),
                               celldm_4 = x / f)
    electrons = ElectronsNamelist(conv_thr=1.e-9)
    ions      = IonsNamelist()
    ...
    return PWInput(control, system, electrons, species, positions, kpoints, ions=ions)
```

</details>

In [ ]:
mono_cases = [
    (f'mono_{i:02d}', build_mono_input(
        A0_ANG, x, ECUTWFC_CONV, NK_ELASTIC,
        prefix=f'mgo_mono_{i:02d}',
        pseudo_dir=PSEUDO_DIR, outdir=ELAST_DIR, pseudos=PSEUDOS,
    ))
    for i, x in enumerate(STRAIN_VALUES)
]

mono_results = runner.run_sweep(mono_cases, ELAST_DIR, force_rerun=FORCE_RERUN)

E_mono  = np.array([r['energy_ry'] * RY_TO_EV for r in mono_results])
E0_mono = E_mono[2]           # x = 0 is at index 2
dE_mono = E_mono - E0_mono

print(f"{'x':>8}  {'E (eV/cell)':>16}  {'ΔE (meV/cell)':>14}")
for x, e, de in zip(STRAIN_VALUES, E_mono, dE_mono):
    print(f'{x:8.4f}  {e:16.6f}  {de*1000:14.4f}')

In [ ]:
mask       = STRAIN_VALUES != 0.0
coeff_mono = fit_elastic_quad(STRAIN_VALUES[mask], dE_mono[mask])
C44        = extract_c44(coeff_mono, V0_CONV_ANG3)

C44_EXP = 155.0   # GPa — experiment (Karki et al. 1997)

print(f"{'':20}  {'C₄₄ (GPa)':>10}")
print(f"{'DFT (this work)':20}  {C44:10.1f}")
print(f"{'Experiment':20}  {C44_EXP:10.1f}")

x_fine = np.linspace(STRAIN_VALUES.min(), STRAIN_VALUES.max(), 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(STRAIN_VALUES[mask], dE_mono[mask] * 1000, zorder=3, label='DFT (relax)')
ax.plot(x_fine, coeff_mono * x_fine**2 * 1000,
        label=f'Fit  $\\alpha = {coeff_mono*1000:.1f}$ meV')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Strain amplitude $x$')
ax.set_ylabel('$\\Delta E$ (meV/cell)')
ax.set_title('MgO — monoclinic shear strain')
ax.legend()
fig.tight_layout()
plt.show()

## How accurate are these results?

| | DFT-PBE (this work) | Experiment (Karki et al. 1997) |
|---|---|---|
| C₁₁ (GPa) | | 297 |
| C₁₂ (GPa) | | 95 |
| C₄₄ (GPa) | | 155 |
| B = (C₁₁+2C₁₂)/3 (GPa) | | ~160 |

DFT-PBE values typically fall 5–10% below experiment for MgO. This is not primarily a numerical error — at the convergence level used here the basis-set and k-mesh contributions are a few GPa at most. The dominant source is the PBE exchange-correlation functional itself: PBE underbinds ionic oxides, slightly overestimating the equilibrium volume and consequently underestimating the curvature of the energy surface in both compression and shear. All three elastic constants and B are shifted in the same direction.

LDA overcorrects in the opposite direction — overbinding gives a smaller lattice parameter and larger elastic constants than experiment. Neither GGA nor LDA is simultaneously correct for both the lattice parameter and the elastic stiffness of MgO; hybrid functionals reduce but do not eliminate the discrepancy.

## Going further: try PBEsol

**PBEsol** is a GGA functional revised for solids and surfaces. It is designed to give better lattice parameters for densely-packed materials like MgO, and in practice it splits the difference between PBE (underbinds) and LDA (overbinds).

You can switch functional without changing pseudopotentials — `pw.x` accepts an `input_dft` keyword in `&SYSTEM` that overrides the XC read from the UPF files. Both `build_ortho_input` and `build_mono_input` forward extra keyword arguments to `SystemNamelist`, so you can simply add `input_dft='PBESOL'` to the calls in the sweep list comprehensions above:

```python
ortho_cases = [
    (f'ortho_pbesol_{i:02d}', build_ortho_input(
        A0_ANG, x, ECUTWFC_CONV, NK_ELASTIC,
        prefix=f'mgo_ortho_pbesol_{i:02d}',
        pseudo_dir=PSEUDO_DIR, outdir=ELAST_DIR, pseudos=PSEUDOS,
        input_dft='PBESOL',
    ))
    for i, x in enumerate(STRAIN_VALUES)
]
```

Re-run both sweeps and compare C₁₁, C₁₂, C₄₄, and B to both the PBE results above and the experimental values. Remember also to redo the EOS fit with PBEsol to get a consistent A0_ANG and B0_GPA before computing the elastic constants.